# 03 · Translate Bubbles

Provide an OpenAI API key, load OCR results, and translate each bubble with ChatGPT.

## 1. Load OpenAI API key
Read the API key from the `OPENAI_API_KEY` environment variable. Set it before running the notebook.

In [1]:
import os

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if OPENAI_API_KEY:
    print('Loaded OpenAI API key from environment. Proceed to the next cells.')
else:
    print('Environment variable OPENAI_API_KEY not set. Export it before continuing.')

Loaded OpenAI API key from environment. Proceed to the next cells.


## 2. Load bubble text from artifacts
Ensure `02-text-ocr.ipynb` ran successfully before this step.

In [2]:
from pathlib import Path
import json

ARTIFACT_DIR = Path('artifacts')
BUBBLES_PATH = ARTIFACT_DIR / 'latest_text_bubble_text.json'

if not BUBBLES_PATH.exists():
    print('Bubble text not found. Run `02-text-ocr.ipynb` to generate OCR results.')
else:
    bubble_payload = json.loads(BUBBLES_PATH.read_text())
    bubbles = bubble_payload.get('bubbles', [])
    print(f'Loaded {len(bubbles)} bubbles.')


Loaded 2 bubbles.


## 3. Translate bubbles with ChatGPT
Send the full bubble set as context so the model can infer language and tone before translating each entry.

In [9]:
FINAL_TRANSLATIONS = []

client = OpenAI(api_key=OPENAI_API_KEY)
context = ''.join(f'{idx+1}. {text}' for idx, text in enumerate(bubbles))
messages = [
  {"role": "system", "content": "You are a translator who keeps the intent, tone, and style of comic dialogue while translating to English."},
  {"role": "user", "content": f"Here are comic speech bubbles. Detect the source language if needed and translate each numbered line to natural English.{context}Reply using JSON with an array named 'translations' that aligns with the input order. First character should be straight {{, no markdown-style json``` prefixes"}
]
try:
  response = client.responses.create(
  model='gpt-4.1-mini',
  input=messages)
except Exception as exc:
  print(f'OpenAI request failed: {exc}')
else:
  output = response.output_text.strip() if hasattr(response, 'output_text') else None
  if not output:
      print('No text returned by the model. Inspect the raw response below:')
      print(response)
  else:
      print('Raw model output:')
      print(output)
      try:
          result = json.loads(output)
          translations = result.get('translations', [])
      except json.JSONDecodeError:
          print('Model output was not valid JSON. Please review the response above.')
          FINAL_TRANSLATIONS = []
      if translations:
          for idx, (original, translated) in enumerate(zip(bubbles, translations), start=1):
              print(f"Bubble #{idx}:\n {translated}")
              FINAL_TRANSLATIONS.append(translated)

Raw model output:
{
  "translations": [
    "I went back alone to my glass room. I gave you everything—my body, my deepest intimacy, sex, sometimes my loneliness. The routine too. But this sadness, I don't think it's exciting or interesting at all. And I don't want to share it with anyone.",
    "I lost him. I feared it from the start. That sooner or later he’d find me too old, meet another woman, younger and more attractive than me. But no, it didn’t go like that. And that hurts even more because it’s all my fault."
  ]
}
Bubble #1:
 I went back alone to my glass room. I gave you everything—my body, my deepest intimacy, sex, sometimes my loneliness. The routine too. But this sadness, I don't think it's exciting or interesting at all. And I don't want to share it with anyone.
Bubble #2:
 I lost him. I feared it from the start. That sooner or later he’d find me too old, meet another woman, younger and more attractive than me. But no, it didn’t go like that. And that hurts even more beca

## 4. Save translated output

In [ ]:
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
payload = {'bubbles': FINAL_TRANSLATIONS}
target_path = ARTIFACT_DIR / 'latest_text_bubble_text_translated.json'
target_path.write_text(json.dumps(payload, indent=2))
print(f'Saved bubble text to {target_path}')